In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.io as sio
from torchvision import models, transforms
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from PIL import Image

In [2]:

device = torch.device("cpu")

# Load AlexNet
alexnet = models.alexnet(weights='IMAGENET1K_V1').to(device).eval()
print(alexnet.features)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (4): ReLU(inplace=True)
  (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU(inplace=True)
  (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU(inplace=True)
  (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
)


In [3]:
# Extract up to Conv5 + ReLU (Index 11)
# Slice [:12] gets layers 0 through 11
features_extractor = nn.Sequential(*list(alexnet.features)[:12])

def extract_and_resize_features(img_path):
    """
    Extract Conv5+ReLU features and interpolate to 227x227 using CPU
    """
    preprocess = transforms.Compose([
        transforms.Resize((227, 227)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    img = Image.open(img_path).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Feature shape: [1, 256, 13, 13]
        raw_features = features_extractor(input_tensor)
        
        # Interpolate to match input resolution 227x227
        interpolated = F.interpolate(raw_features, size=(227, 227), mode='bilinear', align_corners=False)
        
    return interpolated.squeeze(0).numpy() # (256, 227, 227)



In [13]:
# Define parameters
n_images = 59
n_features = 256
pic_dir = "pics"
map_dirs = ["human_map_resized", "box_map_resized", "eye_map_resized"]
map_prefixes = ["human_avg_map_", "box_avg_", "eye_avg_"]

for map_dir, map_prefix in zip(map_dirs, map_prefixes):
    all_X = [] # To store feature matrices
    all_Y = [] # To store target maps
    
    print(f"\n--- Analysis for {map_dir.split('_')[0]} ---")
    print(f"Loading and processing images ...")
    
    for num in range(1, n_images + 1):
        
        
        # Define the paths for the image and the corresponding target map
        img_path = os.path.join(pic_dir, f"testv2_{num}.jpg")
        target_path = os.path.join(map_dir, f"{map_prefix}{num}_resized.npy")
        
        if os.path.exists(img_path) and os.path.exists(target_path):
          # 1. Extract features (Result shape: 256, 227, 227)
          features = extract_and_resize_features(img_path) 
        
          # 2. Load target map (Shape: 227, 227)
          target_map = np.load(target_path)
        
          # 3. Flatten without subsampling to use all pixels
          # x_flat shape: (51529, 256) because 227*227 = 51529
          x_flat = features.reshape(n_features, -1).T
          # y_flat shape: (51529,)
          y_flat = target_map.flatten()
        
          all_X.append(x_flat)
          all_Y.append(y_flat)

    if len(all_X) == n_images:
        print(f"Starting Comparison: wCorr vs SVMregress (Lasso) ...")
    else:
        print(f"Warning: Expected {n_images} images but found {len(all_X)}. Check file paths.")
        break

    # Start comparison of wCorr and LassoCV 
    loo_cc_wCorr = []
    loo_cc_Lasso = []

    for i in range(n_images):
        # 1. Prepare Data
        X_train = np.concatenate([all_X[j] for j in range(len(all_X)) if j != i], axis=0)
        y_train = np.concatenate([all_Y[j] for j in range(len(all_Y)) if j != i], axis=0)
    
        X_test = all_X[i]
        y_test = all_Y[i]
    
        # 2. Scaling (Essential for Lasso)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
    
        # --- Method A: wCorr (Pearson Correlation) ---
        # Independent per-channel correlation
        wcorr_weights = np.array([np.corrcoef(X_train_scaled[:, k], y_train)[0, 1] for k in range(n_features)])
        y_pred_wcorr = X_test_scaled @ wcorr_weights
        cc_wcorr = np.corrcoef(y_pred_wcorr, y_test)[0, 1]
        loo_cc_wCorr.append(cc_wcorr)
    
        # --- Method B: SVMregress (Lasso with 10-fold CV) ---
         # Sampling 10% for speed (approx 300,000 pixels)
        sample_size = int(len(X_train_scaled) * 0.1)
        idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
    
        # LassoCV automatically handles the 10-fold CV
        # n_jobs=-1 uses all CPU cores
        lasso = LassoCV(cv=10, max_iter=2000, n_jobs=-1).fit(X_train_scaled[idx], y_train[idx])
        lasso_weights = lasso.coef_
    
        y_pred_lasso = X_test_scaled @ lasso_weights
        cc_lasso = np.corrcoef(y_pred_lasso, y_test)[0, 1]
        loo_cc_Lasso.append(cc_lasso)

    # --- Final Stats ---
    print(f"\n--- Results for {map_dir.split('_')[0]} ---")
    print(f"wCorr Mean CC: {np.mean(loo_cc_wCorr):.4f}")
    print(f"Lasso Mean CC: {np.mean(loo_cc_Lasso):.4f}")
    


--- Analysis for human ---
Loading and processing images ...
Starting Comparison: wCorr vs SVMregress (Lasso) ...

--- Results for human ---
wCorr Mean CC: 0.5148
Lasso Mean CC: 0.5110

--- Analysis for box ---
Loading and processing images ...
Starting Comparison: wCorr vs SVMregress (Lasso) ...

--- Results for box ---
wCorr Mean CC: 0.3727
Lasso Mean CC: 0.3947

--- Analysis for eye ---
Loading and processing images ...
Starting Comparison: wCorr vs SVMregress (Lasso) ...

--- Results for eye ---
wCorr Mean CC: 0.3978
Lasso Mean CC: 0.3880


In [7]:
# ---- Get the weights using wCorr method ----
n_images = 59
n_features = 256
pic_dir = "pics"
map_dirs = ["human_map_resized", "box_map_resized", "eye_map_resized"]
map_prefixes = ["human_avg_map_", "box_avg_", "eye_avg_"]
final_weights_dict = {}

for map_dir, map_prefix in zip(map_dirs, map_prefixes):
    all_X = [] # To store feature matrices
    all_Y = [] # To store target maps
    
    current_name = map_dir.split('_')[0]
    print(f"\n--- Analysis for {current_name} ---")
    print(f"Loading and processing images ...")
    
    for num in range(1, n_images + 1):
        # Define paths
        img_path = os.path.join(pic_dir, f"testv2_{num}.jpg")
        target_path = os.path.join(map_dir, f"{map_prefix}{num}_resized.npy")
        
        if os.path.exists(img_path) and os.path.exists(target_path):
            # 1. Extract features (Result shape: 256, 227, 227)
            features = extract_and_resize_features(img_path) 
            # 2. Load target map (Shape: 227, 227)
            target_map = np.load(target_path)
            # 3. Flatten without subsampling
            x_flat = features.reshape(n_features, -1).T
            y_flat = target_map.flatten()
        
            all_X.append(x_flat)
            all_Y.append(y_flat)

    if len(all_X) == n_images:
        print(f"Starting calculation for {current_name}...")
    else:
        print(f"Warning: Expected {n_images} images but found {len(all_X)}.")
        break

    # To store weights of each individual image
    weights_list = []

    for i in range(n_images):
        # Calculate weights for the i-th image independently (Method A logic)
        img_w = np.array([np.corrcoef(all_X[i][:, k], all_Y[i])[0, 1] for k in range(n_features)])
        weights_list.append(img_w)
    
    # Calculate the final average weights for this map type
    avg_weights = np.nanmean(weights_list, axis=0)
    final_weights_dict[current_name] = avg_weights

    print(f"--- Done for {current_name} ---")


--- Analysis for human ---
Loading and processing images ...
Starting calculation for human...
--- Done for human ---

--- Analysis for box ---
Loading and processing images ...
Starting calculation for box...
--- Done for box ---

--- Analysis for eye ---
Loading and processing images ...
Starting calculation for eye...
--- Done for eye ---
